# Model Comparison — Normal vs Student-t × Constant vs GARCH

This notebook compares the four Monte Carlo simulation configurations available in RiskLens:

| Config | Shocks | Volatility | Description |
|--------|--------|------------|-------------|
| **Baseline** | Normal | Constant | Textbook GBM |
| **Fat tails** | Student-t | Constant | Heavier tails, flat vol |
| **Vol clustering** | Normal | GARCH(1,1) | Time-varying vol, thin tails |
| **Full model** | Student-t | GARCH(1,1) | Vol clustering + fat tails |

We run all four on the same asset and seed, then compare VaR, CVaR, path spreads, and final price distributions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data.fetch import fetch_asset_data
from src.data.process import clean_market_data, add_returns
from src.analytics.monte_carlo import (
    simulate_paths,
    simulation_summary,
    compute_var,
    compute_cvar,
    fit_t_distribution,
    fit_garch,
)

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Data & Configuration

In [ ]:
TICKER = "BTC-USD"
N_DAYS = 252
N_SIMS = 10_000
CONFIDENCE = 0.95
SEED = 42

raw = fetch_asset_data(TICKER)
df = clean_market_data(raw)
df = add_returns(df)
returns = df["returns"]
close = df["close"]

print(f"Loaded {len(df)} trading days for {TICKER}")
print(f"Simulation: {N_SIMS:,} paths × {N_DAYS} days, seed={SEED}")

# Pre-fit models
t_info = fit_t_distribution(returns)
garch_info = fit_garch(returns)

print(f"\nStudent-t fit: df = {t_info['df']:.2f} ({t_info['tail_description']})")
print(f"GARCH(1,1) fit: α = {garch_info['alpha']:.4f}, β = {garch_info['beta']:.4f}, "
      f"persistence = {garch_info['persistence']:.4f}, long-run vol = {garch_info['long_run_vol']:.2%}")

## 2. Run All Four Models

In [ ]:
MODELS = {
    "Normal + Constant": dict(distribution="normal", volatility_model="constant"),
    "Student-t + Constant": dict(distribution="t", volatility_model="constant", df_t=t_info["df"]),
    "Normal + GARCH": dict(distribution="normal", volatility_model="garch", garch_params=garch_info),
    "Student-t + GARCH": dict(distribution="t", volatility_model="garch", df_t=t_info["df"], garch_params=garch_info),
}

results = {}
initial_price = close.iloc[-1]

for name, params in MODELS.items():
    paths = simulate_paths(close, returns, n_days=N_DAYS, n_simulations=N_SIMS, seed=SEED, **params)
    fp = paths.iloc[-1]
    summary_95 = simulation_summary(fp, initial_price, confidence=CONFIDENCE)
    summary_99 = simulation_summary(fp, initial_price, confidence=0.99)
    results[name] = {
        "paths": paths,
        "final_prices": fp,
        "summary": summary_95,
        "var": summary_95["var"],
        "cvar": summary_95["cvar"],
        "var_99": summary_99["var"],
        "cvar_99": summary_99["cvar"],
    }

print("All four models simulated.")

## 3. Summary Table

Side-by-side comparison of key risk metrics across all four configurations.

In [ ]:
rows = []
for name, r in results.items():
    s = r["summary"]
    rows.append({
        "Model": name,
        "VaR (95%)": f"{s['var']:.2%}",
        "CVaR (95%)": f"{s['cvar']:.2%}",
        "P(Gain)": f"{s['prob_gain']:.1%}",
        "P(Loss)": f"{s['prob_loss']:.1%}",
        "Mean Price": f"${s['mean_final_price']:,.0f}",
        "Median Price": f"${s['median_final_price']:,.0f}",
        "Min Price": f"${s['min_final_price']:,.0f}",
        "Max Price": f"${s['max_final_price']:,.0f}",
    })

comparison = pd.DataFrame(rows).set_index("Model")
comparison

## 4. VaR & CVaR Comparison

Bar chart showing how tail risk estimates change across models. More negative = more conservative (realistic).

In [ ]:
model_names = list(results.keys())
x = np.arange(len(model_names))
width = 0.35

for conf, var_key, cvar_key in [("95%", "var", "cvar"), ("99%", "var_99", "cvar_99")]:
    vars_ = [results[m][var_key] for m in model_names]
    cvars_ = [results[m][cvar_key] for m in model_names]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, vars_, width, label=f"VaR ({conf})", color="#ff7f0e")
    bars2 = ax.bar(x + width/2, cvars_, width, label=f"CVaR ({conf})", color="#d62728")

    ax.set_ylabel("Return (%)")
    ax.set_title(f"{TICKER} — VaR & CVaR {conf} by Model ({N_DAYS} days)")
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=15, ha="right")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend()
    ax.axhline(0, color="gray", linewidth=0.5)

    for bars in [bars1, bars2]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
                    f"{bar.get_height():.1%}", ha="center", va="top", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 5. Final Price Distributions

Overlaid KDE of the full distribution, then three zoomed views: left tail (worst 5%), central body (middle 90%), and right tail (best 5%).

In [ ]:
colors = {"Normal + Constant": "#1f77b4", "Student-t + Constant": "#ff7f0e",
          "Normal + GARCH": "#2ca02c", "Student-t + GARCH": "#d62728"}

# --- Full distribution ---
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    sns.kdeplot(r["final_prices"], ax=ax, label=name, color=colors[name], linewidth=1.5)
ax.axvline(initial_price, color="gray", linewidth=1.5, linestyle=":", label=f"Initial: ${initial_price:,.0f}")
ax.set_title(f"{TICKER} — Final Price Distribution (full)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

# --- Compute percentile bounds per model ---
bounds = {}
for name, r in results.items():
    fp = r["final_prices"]
    bounds[name] = {"p5": np.percentile(fp, 5), "p95": np.percentile(fp, 95)}

# Global bounds for consistent x-axis across models
left_max = max(b["p5"] for b in bounds.values()) * 1.05
right_min = min(b["p95"] for b in bounds.values()) * 0.95

# --- Left tail (worst 5%) ---
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    tail = r["final_prices"][r["final_prices"] <= bounds[name]["p5"]]
    sns.kdeplot(tail, ax=ax, label=name, color=colors[name], linewidth=1.5, clip=(0, left_max))
ax.set_xlim(right=left_max)
ax.axvline(initial_price, color="gray", linewidth=1, linestyle=":")
ax.set_title(f"{TICKER} — Left Tail (worst 5%)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

# --- Central body (middle 90%) ---
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    fp = r["final_prices"]
    center = fp[(fp > bounds[name]["p5"]) & (fp < bounds[name]["p95"])]
    sns.kdeplot(center, ax=ax, label=name, color=colors[name], linewidth=1.5)
ax.axvline(initial_price, color="gray", linewidth=1, linestyle=":")
ax.set_title(f"{TICKER} — Central Body (middle 90%)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

# --- Right tail (best 5%) ---
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    tail = r["final_prices"][r["final_prices"] >= bounds[name]["p95"]]
    sns.kdeplot(tail, ax=ax, label=name, color=colors[name], linewidth=1.5)
ax.set_xlim(left=right_min)
ax.axvline(initial_price, color="gray", linewidth=1, linestyle=":")
ax.set_title(f"{TICKER} — Right Tail (best 5%)")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Monte Carlo Paths — Side by Side

200 sample paths per model. Compare the spread and shape of the fan.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, (name, r) in zip(axes.flat, results.items()):
    paths = r["paths"]
    sample = paths.iloc[:, :200]
    ax.plot(sample, color=colors[name], alpha=0.03, linewidth=0.5)
    ax.plot(paths.median(axis=1), color="white", linewidth=1.5, label="Median")
    ax.plot(paths.quantile(0.05, axis=1), color="black", linewidth=1, linestyle="--", label="5th pctl")
    ax.plot(paths.quantile(0.95, axis=1), color="black", linewidth=1, linestyle="--", label="95th pctl")
    ax.axhline(initial_price, color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(name)
    ax.legend(loc="upper left", fontsize=8)

axes[1, 0].set_xlabel("Trading Day")
axes[1, 1].set_xlabel("Trading Day")
axes[0, 0].set_ylabel("Price (USD)")
axes[1, 0].set_ylabel("Price (USD)")

fig.suptitle(f"{TICKER} — Monte Carlo Paths ({N_SIMS:,} sims, {N_DAYS} days)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Scenario Bucket Comparison

How each model distributes outcomes across the five scenario categories.

In [ ]:
from src.analytics.monte_carlo import scenario_buckets

bucket_labels = ["Severe Loss\n(<-30%)", "Moderate Loss\n(-30% to -10%)", "Flat\n(-10% to +10%)",
                 "Moderate Gain\n(+10% to +30%)", "Strong Gain\n(>+30%)"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
bucket_colors = ["#d62728", "#ff7f0e", "#7f7f7f", "#2ca02c", "#1f77b4"]

for ax, (name, r) in zip(axes.flat, results.items()):
    buckets = scenario_buckets(r["final_prices"], initial_price)
    vals = list(buckets.values())
    bars = ax.bar(bucket_labels, vals, color=bucket_colors, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.1%}", ha="center", fontsize=8, fontweight="bold")
    ax.set_title(name)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.tick_params(axis="x", labelsize=7)

fig.suptitle(f"{TICKER} — Scenario Distribution by Model", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Tail Risk Deep Dive

Zooming into the left tail: the 1st and 5th percentiles of final returns, showing how much more risk the advanced models reveal.

In [ ]:
tail_rows = []
for name, r in results.items():
    rets = (r["final_prices"] - initial_price) / initial_price
    tail_rows.append({
        "Model": name,
        "1st pctl": f"{np.percentile(rets, 1):.2%}",
        "5th pctl (VaR 95%)": f"{np.percentile(rets, 5):.2%}",
        "10th pctl (VaR 90%)": f"{np.percentile(rets, 10):.2%}",
        "Worst case": f"{rets.min():.2%}",
        "Best case": f"{rets.max():.2%}",
        "Kurtosis": f"{float(pd.Series(rets).kurtosis()):.2f}",
        "Skewness": f"{float(pd.Series(rets).skew()):.2f}",
    })

tail_df = pd.DataFrame(tail_rows).set_index("Model")
tail_df

## 9. Key Takeaways

- **Normal + Constant** (baseline GBM) underestimates tail risk — thinnest tails, lowest kurtosis.
- **Student-t** fattens both tails symmetrically — worst-case and best-case become more extreme.
- **GARCH** reshapes the distribution based on the current volatility regime — if we're in a high-vol period, the fan is wider.
- **Student-t + GARCH** combines both effects — the most realistic model for financial data.
- The difference in **CVaR** between baseline and full model is the risk you miss by using textbook assumptions.